In [1]:
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
import os
from datetime import datetime

os.environ['TZ'] = 'Europe/Berlin'
import time
time.tzset()

def checkpoint(msg):
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {msg}", flush=True)

In [2]:

OUT_DIR = "/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/05_significant_MP/05_violins_per_MP_celltype"
os.makedirs(OUT_DIR, exist_ok=True)

adata = sc.read_h5ad("/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/data/01_extGBmap/immune_cells_extended_gbmap_hgnc.h5ad")
mp_scores = pd.read_csv("/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/04_NMF_GSEAonHGNC/04_nmf_all_gsea/mp_scores.csv", index_col=0)


In [3]:
for col in mp_scores.columns:
    if col not in adata.obs.columns:
        adata.obs[col] = mp_scores[col].reindex(adata.obs_names).values

mp_score_cols = mp_scores.columns.tolist()
checkpoint(f"Found {len(mp_score_cols)} MP score columns")

n_mps = len(mp_score_cols)
n_cols = 5
n_rows = -(-n_mps // n_cols)  # ceil division

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4, n_rows * 4))
axes = axes.flatten()

for i, mp_col in enumerate(mp_score_cols):
    mp_name = mp_col.replace("_score", "")
    sc.pl.violin(adata, keys=mp_col, groupby="annotation_level_3", rotation=45, show=False, ax=axes[i], stripplot=False)
    axes[i].set_title(mp_name, fontsize=10)
    axes[i].set_xlabel("")
    axes[i].set_ylabel(f"{mp_name} score", fontsize=8)

for j in range(n_mps, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "violin_all_MPs_per_celltype_compressed.pdf"), dpi=120, bbox_inches="tight")
plt.close()
checkpoint("Combined PDF with all MP violin plots saved")

[2026-07-15 13:53:04] Found 30 MP score columns
[2026-07-15 13:54:19] Combined PDF with all MP violin plots saved
